In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, brier_score_loss, precision_recall_curve, auc
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import label_binarize

import xgboost as xgb
import lightgbm as lgb

print("✅ Môi trường đã sẵn sàng!")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")


✅ Môi trường đã sẵn sàng!
XGBoost version: 3.2.0
LightGBM version: 4.6.0


> **Fix note (2026-08-04):** this notebook previously overwrote the real ALPI/PIADE data loaded in Cell 0 with a synthetically generated table (`load_or_generate_alpi_data()`, `np.random`-based, 20,000 rows) whenever `./alpi_data/alpi_processed.csv` was missing. All results below (macro-F1, AUPRC, accuracy for XGBoost/LightGBM/Qwen1.5-0.5B/Chronos-T5) were computed on that fake table, not on the real 429,394-row dataset. The fallback has been removed; the pipeline now only runs on the real data loaded and mapped in Cells 0-1. **This notebook has not yet been re-executed on Kaggle against real data** — all outputs below are stale/cleared until that re-run happens. See `reports/ALPI_BASELINE_BENCHMARK_REPORT.md` for the invalidated-numbers note.

# 0. EDA

In [3]:
# ==============================================================================
# CELL 0: NẠP DỮ LIỆU THẬT ALPI / PIADE DATASET (STRICT REAL DATA LOADING)
# ==============================================================================
import os
import pandas as pd
import numpy as np

print("==============================================================================")
print("🔍 NẠP VÀ KIỂM TRA BẢNG DỮ LIỆU THẬT ALPI / PIADE (RAW DATASET)")
print("==============================================================================")

RAW_DATA_PATH = '/kaggle/input/datasets/orvile/packaging-industry-anomaly-detection-dataset/raw_data.csv'

# 🛑 KIỂM TRA TỆP DỮ LIỆU THẬT
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(
        f"\n❌ CHƯA TÌM THẤY FILE: {RAW_DATA_PATH}\n"
        "👉 Vui lòng nhấn '+ Add Input' ở góc phải Kaggle Notebook và thêm dataset: 'packaging-industry-anomaly-detection-dataset'"
    )

# 1. Nạp file CSV dữ liệu thô thật
df_raw = pd.read_csv(RAW_DATA_PATH, encoding='ascii', delimiter=',')

# 2. Chuyển đổi cột thời gian 'interval_start' sang định dạng Chuẩn Datetime
if 'interval_start' in df_raw.columns:
    df_raw['interval_start'] = pd.to_datetime(df_raw['interval_start'], errors='coerce')
    df_raw = df_raw.sort_values(by='interval_start').reset_index(drop=True)

# ------------------------------------------------------------------------------
# 📊 THỐNG KÊ TỔNG QUAN TỆP THẬT
# ------------------------------------------------------------------------------
print(f"✅ NẠP THÀNH CÔNG DỮ LIỆU THẬT ALPI/PIADE!")
print(f" • Kích thước tệp thô : {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
if 'interval_start' in df_raw.columns:
    print(f" • Thời gian bắt đầu : {df_raw['interval_start'].min()}")
    print(f" • Thời gian kết thúc: {df_raw['interval_start'].max()}")

print("\n📋 Xem trước 5 dòng đầu tiên của dữ liệu thô (Head):")
display(df_raw.head()) if 'display' in globals() else print(df_raw.head())
print("==============================================================================")


🔍 NẠP VÀ KIỂM TRA BẢNG DỮ LIỆU THẬT ALPI / PIADE (RAW DATASET)
✅ NẠP THÀNH CÔNG DỮ LIỆU THẬT ALPI/PIADE!
 • Kích thước tệp thô : 429,394 dòng x 10 cột
 • Thời gian bắt đầu : 2020-01-01 00:05:02.863000+00:00
 • Thời gian kết thúc: 2022-01-01 23:41:35.677000+00:00

📋 Xem trước 5 dòng đầu tiên của dữ liệu thô (Head):
                    interval_start equipment_ID  alarm              type  \
0 2020-01-01 00:05:02.863000+00:00          s_4  A_000        production   
1 2020-01-01 00:05:05.743000+00:00          s_4  A_000  performance_loss   
2 2020-01-01 00:08:36.983000+00:00          s_4  A_000        production   
3 2020-01-01 00:08:47.734000+00:00          s_4  A_000  performance_loss   
4 2020-01-01 00:13:03.273000+00:00          s_4  A_000        production   

          start           end  elapsed     pi     po  speed  
0  1.577837e+09  1.577837e+09     2880  92505  92241   6500  
1  1.577837e+09  1.577837e+09   211240  92509  92245   6500  
2  1.577837e+09  1.577837e+09    10751  9

# 1. Load data

In [ ]:
# ==============================================================================
# CELL 1: CHUAN HOA DU LIEU THAT SANG SCHEMA MO HINH
# (timestamp, machine_id, alarm_code, machine_state)
# ==============================================================================
# FIX: cell nay truoc day goi load_or_generate_alpi_data(), mot ham am tham
# GHI DE df_raw THAT (429,394 dong, nap o Cell 0) bang mot bang np.random gia
# (20,000 dong) moi khi ./alpi_data/alpi_processed.csv chua ton tai. Toan bo
# ket qua benchmark truoc day (XGBoost/LightGBM/Qwen1.5-0.5B/Chronos-T5) duoc
# huan luyen/danh gia tren du lieu GIA, khong phai ALPI/PIADE that. Ham sinh
# du lieu gia da bi xoa hoan toan; pipeline ben duoi chi chay tren df_raw that.

REQUIRED_RAW_COLUMNS = {"interval_start", "equipment_ID", "alarm", "type"}
missing_cols = REQUIRED_RAW_COLUMNS - set(df_raw.columns)
if missing_cols:
    raise KeyError(
        f"df_raw that dang thieu cac cot bat buoc: {missing_cols}. "
        "Kiem tra lai schema cua raw_data.csv truoc khi tiep tuc."
    )

print("Cac gia tri duy nhat trong cot 'type' (dung de suy ra machine_state):")
print(df_raw["type"].value_counts())

# Anh xa OEE loss category -> machine_state (0: Production, 1: Idle/Performance
# loss, 2: Stop/Downtime), theo quy uoc OEE chuan cua PIADE. XAC MINH danh sach
# category in ra o tren khop voi anh xa ben duoi truoc khi tin tuong ket qua;
# neu thieu category, dong ben duoi se raise loi thay vi am tham bo qua.
TYPE_TO_STATE = {
    "production": 0,
    "performance_loss": 1,
    "availability_loss": 2,
    "quality_loss": 1,
}
unmapped = set(df_raw["type"].dropna().unique()) - set(TYPE_TO_STATE)
if unmapped:
    raise ValueError(
        f"Cot 'type' co gia tri chua duoc anh xa sang machine_state: {unmapped}. "
        "Cap nhat TYPE_TO_STATE truoc khi chay tiep."
    )

df_model = pd.DataFrame({
    "timestamp": df_raw["interval_start"],
    "machine_id": df_raw["equipment_ID"],
    "alarm_code": df_raw["alarm"],
    "machine_state": df_raw["type"].map(TYPE_TO_STATE),
}).dropna(subset=["machine_state"]).reset_index(drop=True)
df_model["machine_state"] = df_model["machine_state"].astype(int)

print(f"Da chuan hoa {len(df_model):,} dong du lieu THAT sang schema mo hinh.")
df_raw = df_model
df_raw.head()


# 2. Feature Engineering

In [ ]:
def create_window_features(df, window_size=5):
    print("⚙️ Đang trích xuất đặc trưng Cửa sổ 60s (Sliding Window)...")
    df = df.copy()
    df["alarm_code_cat"] = df["alarm_code"].astype("category").cat.codes
    
    features, labels_b1, labels_b5 = [], [], []
    alarm_codes_encoded = df["alarm_code_cat"].values
    states = df["machine_state"].values
    
    for i in range(window_size, len(df) - 1):
        window_alarms = alarm_codes_encoded[i-window_size:i]
        window_states = states[i-window_size:i]
        
        feat = {
            "last_alarm": window_alarms[-1],
            "alarm_count": len(window_alarms),
            "unique_alarms": len(np.unique(window_alarms)),
            "most_freq_alarm": pd.Series(window_alarms).mode()[0] if len(window_alarms) > 0 else 0,
            "state_mode": pd.Series(window_states).mode()[0] if len(window_states) > 0 else 0,
        }
        features.append(feat)
        labels_b1.append(states[i])
        labels_b5.append(alarm_codes_encoded[i+1])
        
    X = pd.DataFrame(features)
    return X, np.array(labels_b1), np.array(labels_b5)

X, y_b1, y_b5 = create_window_features(df_raw)


# 3. Train, test model XGBosst và LightBM cho task B1

In [ ]:
print("\n--- 🚀 THỰC NGHIỆM TASK B1: STATE ESTIMATION (SUPERVISED TRAINED BASELINE) ---")
X_train, X_test, y_train_b1, y_test_b1 = train_test_split(X, y_b1, test_size=0.2, random_state=42, stratify=y_b1)

# 1. XGBoost Classifier (SUPERVISED TRAINED)
xgb_b1 = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_b1.fit(X_train, y_train_b1)
y_pred_xgb = xgb_b1.predict(X_test)
y_prob_xgb = xgb_b1.predict_proba(X_test)

f1_xgb_b1 = f1_score(y_test_b1, y_pred_xgb, average='macro')
brier_xgb_b1 = np.mean([brier_score_loss((y_test_b1 == c).astype(int), y_prob_xgb[:, c]) for c in range(y_prob_xgb.shape[1])])
print(f"✅ [XGBoost TRAINED] Task B1 - Macro-F1: {f1_xgb_b1:.4f} | Brier Score: {brier_xgb_b1:.4f}")

# 2. LightGBM Classifier (SUPERVISED TRAINED)
lgb_b1 = lgb.LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)
lgb_b1.fit(X_train, y_train_b1)
y_pred_lgb = lgb_b1.predict(X_test)
y_prob_lgb = lgb_b1.predict_proba(X_test)

f1_lgb_b1 = f1_score(y_test_b1, y_pred_lgb, average='macro')
brier_lgb_b1 = np.mean([brier_score_loss((y_test_b1 == c).astype(int), y_prob_lgb[:, c]) for c in range(y_prob_lgb.shape[1])])
print(f"✅ [LightGBM TRAINED] Task B1 - Macro-F1: {f1_lgb_b1:.4f} | Brier Score: {brier_lgb_b1:.4f}")


# 4. Train, test cho task B5

In [ ]:
print("\n--- 🚀 THỰC NGHIỆM TASK B5: NEXT ALARM PREDICTION (SUPERVISED TRAINED BASELINE) ---")
X_train_b5, X_test_b5, y_train_b5, y_test_b5 = train_test_split(X, y_b5, test_size=0.2, random_state=42)

# 1. XGBoost
xgb_b5 = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_b5.fit(X_train_b5, y_train_b5)
y_prob_xgb_b5 = xgb_b5.predict_proba(X_test_b5)

n_classes = len(np.unique(y_b5))
y_test_bin = label_binarize(y_test_b5, classes=range(n_classes))
precision, recall, _ = precision_recall_curve(y_test_bin.ravel(), y_prob_xgb_b5.ravel())
auprc_xgb_b5 = auc(recall, precision)
print(f"✅ [XGBoost TRAINED] Task B5 - AUPRC: {auprc_xgb_b5:.4f}")

# 2. LightGBM
lgb_b5 = lgb.LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1)
lgb_b5.fit(X_train_b5, y_train_b5)
y_prob_lgb_b5 = lgb_b5.predict_proba(X_test_b5)

precision_lgb, recall_lgb, _ = precision_recall_curve(y_test_bin.ravel(), y_prob_lgb_b5.ravel())
auprc_lgb_b5 = auc(recall_lgb, precision_lgb)
print(f"✅ [LightGBM TRAINED] Task B5 - AUPRC: {auprc_lgb_b5:.4f}")


# 5. Tổng hợp kết quả

In [ ]:
print("\n==============================================================================")
print("📌 THỐNG KÊ KẾT QUẢ BASELINE (MÔ HÌNH CÓ TRAIN - KHÔNG PHẢI ZERO-SHOT)")
print("==============================================================================")

results_summary = pd.DataFrame([
    {"Task": "B1 (State Est)", "Model": "XGBoost", "Type": "Supervised Train", "Macro-F1/AUPRC": f"{f1_xgb_b1:.4f}", "Brier Score": f"{brier_xgb_b1:.4f}"},
    {"Task": "B1 (State Est)", "Model": "LightGBM", "Type": "Supervised Train", "Macro-F1/AUPRC": f"{f1_lgb_b1:.4f}", "Brier Score": f"{brier_lgb_b1:.4f}"},
    {"Task": "B5 (Next Alarm)", "Model": "XGBoost", "Type": "Supervised Train", "Macro-F1/AUPRC": f"{auprc_xgb_b5:.4f}", "Brier Score": "N/A"},
    {"Task": "B5 (Next Alarm)", "Model": "LightGBM", "Type": "Supervised Train", "Macro-F1/AUPRC": f"{auprc_lgb_b5:.4f}", "Brier Score": "N/A"},
])

print(results_summary.to_string(index=False))


# 6. Chuyển đổi dữ liệu từ table thành text cho LLM

In [ ]:
# ==============================================================================
# CELL 7: Chuyển đổi dữ liệu ALPI sang dạng Text (Serialize) cho LLM
# ==============================================================================
def serialize_alpi_to_text(df, window_size=5):
    print("📝 Đang Serialize dữ liệu ALPI thành Prompt văn bản cho LLM...")
    df["alarm_code_cat"] = df["alarm_code"].astype("category").cat.codes
    
    text_data = []
    for i in range(window_size, len(df) - 1):
        window_df = df.iloc[i-window_size:i]
        target_state = df.iloc[i]["machine_state"]
        target_alarm = df.iloc[i+1]["alarm_code_cat"]
        
        # Tạo chuỗi mô tả lịch sử 60s
        history_str = ", ".join([f"Alarm {row['alarm_code']} at state {row['machine_state']}" for _, row in window_df.iterrows()])
        
        prompt = f"Industrial Log History (last 60s): [{history_str}]. Predict current machine state (0: Production, 1: Idle, 2: Stop) and next alarm code."
        response = f"State: {target_state}, Next Alarm: {target_alarm}"
        
        text_data.append({"prompt": prompt, "response": response, "full_text": f"Instruction: {prompt}\nResponse: {response}"})
        
    return pd.DataFrame(text_data)

df_text = serialize_alpi_to_text(df_raw)

# Time-based Split (80% Train đầu, 20% Test cuối)
train_size = int(len(df_text) * 0.8)
train_text_df = df_text.iloc[:train_size]
test_text_df = df_text.iloc[train_size:]

print(f"✅ Đã tạo {len(df_text)} câu Instruction. Tập Train: {len(train_text_df)}, Tập Test: {len(test_text_df)}")
print("\nVí dụ 1 mẫu Prompt:")
print(train_text_df.iloc[0]["full_text"])


# 7. Train LLM

In [ ]:
# ==============================================================================
# CELL 8: Train/Fine-tune LLM thuần văn bản (Trainable Text-LLM Baseline)
# ==============================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset

print("\n--- 🚀 HUẤN LUYỆN MODEL LLM THUẦN VĂN BẢN (TRAINED TEXT-LLM BASELINE) ---")

MODEL_NAME = "Qwen/Qwen1.5-0.5B"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model với float32 hoặc bfloat16 để tránh lỗi FP16 gradient unscaling
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None
)

# Chuyển dataframe sang HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_text_df[["full_text"]])
test_dataset = Dataset.from_pandas(test_text_df[["full_text"]])

def tokenize_function(examples):
    tokens = tokenizer(examples["full_text"], padding="max_length", truncation=True, max_length=128)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Thiết lập tham số huấn luyện (Đã tắt fp16 để đảm bảo tính ổn định tuyệt đối trên T4)
training_args = TrainingArguments(
    output_dir="./llm_alpi_results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=50,
    save_strategy="no",
    eval_strategy="epoch",
    fp16=False,                         # Tắt FP16 hỗn hợp để tránh lỗi gradient scale
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

print("⏳ Bắt đầu huấn luyện LLM trên tập dữ liệu ALPI Text...")
trainer.train()
print("✅ Huấn luyện LLM thành công! Đã sẵn sàng làm mốc so sánh với Cosmos Zero-shot.")


# 8. Đánh giá LLM

In [ ]:
# ==============================================================================
# CELL 9: Đánh giá & In kết quả riêng của Fine-tuned LLM
# ==============================================================================
import re
from tqdm import tqdm
from sklearn.metrics import f1_score

print("\n--- 🔍 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH FINE-TUNED LLM TRÊN TẬP TEST ---")

model.eval()

y_true_b1, y_pred_b1 = [], []
y_true_b5, y_pred_b5 = [], []

# Lấy 500 mẫu đại diện từ tập Test để đánh giá tốc độ cao
test_subset = test_text_df.head(500)

print("⏳ Đang suy luận (Inference) từ LLM...")
for idx, row in tqdm(test_subset.iterrows(), total=len(test_subset)):
    prompt_text = f"Instruction: {row['prompt']}\nResponse:"
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)
        
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Trích xuất State và Next Alarm từ đầu ra văn bản của LLM bằng Regex
    match_b1 = re.search(r"State:\s*(\d+)", generated_text)
    match_b5 = re.search(r"Next Alarm:\s*(\d+)", generated_text)
    
    # Thực tế (Ground truth)
    gt_b1 = int(re.search(r"State:\s*(\d+)", row["response"]).group(1))
    gt_b5 = int(re.search(r"Next Alarm:\s*(\d+)", row["response"]).group(1))
    
    y_true_b1.append(gt_b1)
    y_pred_b1.append(int(match_b1.group(1)) if match_b1 else 0)
    
    y_true_b5.append(gt_b5)
    y_pred_b5.append(int(match_b5.group(1)) if match_b5 else 0)

# Tính toán các chỉ số
f1_llm_b1 = f1_score(y_true_b1, y_pred_b1, average="macro")
acc_llm_b5 = np.mean(np.array(y_true_b5) == np.array(y_pred_b5))

print("\n==============================================================================")
print("📌 KẾT QUẢ CỦA FINE-TUNED LLM (QWEN1.5-0.5B - TRAINED TEXT-LLM)")
print("==============================================================================")
print(f"🔹 Task B1 (State Estimation)   - Macro-F1 Score : {f1_llm_b1:.4f}")
print(f"🔹 Task B5 (Next Alarm Prediction) - Exact Accuracy : {acc_llm_b5:.4f}")
print("==============================================================================")


# 9. Cài đặt Chronos

In [ ]:
# ==============================================================================
# CELL 10: Cài đặt thư viện Chronos Forecasting (Amazon Science)
# ==============================================================================
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

print("✅ Đã cài đặt thành công thư viện Chronos Forecasting!")


# 10. train và đánh giá Chronos 2

In [ ]:
# ==============================================================================
# CELL 11: Nạp Mô hình Chronos-T5 & Đánh giá trên Dữ liệu Chuỗi Thời Gian ALPI
# ==============================================================================
import torch
import numpy as np
import pandas as pd
from chronos import ChronosPipeline
from tqdm import tqdm

print("\n--- 🚀 THỰC NGHIỆM MÔ HÌNH TIME-SERIES FOUNDATION MODEL (AMAZON CHRONOS T5) ---")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang tải mô hình Amazon Chronos-T5 trên thiết bị: {device}...")

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map=device,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
)

# 1. Chuẩn bị dữ liệu dạng Chuỗi thời gian liên tục (Continuous Time Series)
df_chronos = df_raw.copy()
df_chronos["alarm_code_cat"] = df_chronos["alarm_code"].astype("category").cat.codes

test_series_list = []
y_true_chronos_b5 = []

window_size = 30  # Nhìn 30 sự kiện lỗi quá khứ để dự đoán sự kiện tiếp theo
for i in range(len(df_chronos) - window_size - 1):
    if len(test_series_list) >= 500:
        break
    context_tensor = torch.tensor(df_chronos["alarm_code_cat"].iloc[i:i+window_size].values, dtype=torch.float32)
    test_series_list.append(context_tensor)
    y_true_chronos_b5.append(df_chronos["alarm_code_cat"].iloc[i+window_size])

print(f"⏳ Đang thực hiện dự báo Zero-shot với Chronos T5 trên {len(test_series_list)} chuỗi thời gian ALPI...")

# 2. Thực hiện dự báo (Forecast next step) - Đã sửa tham số inputs chuẩn API Chronos
prediction_length = 1
y_pred_chronos_b5 = []

for context in tqdm(test_series_list):
    # Truyền trực tiếp context làm vị trí đầu tiên (inputs)
    forecast = pipeline.predict(
        context,
        prediction_length=prediction_length,
        num_samples=20,
    )
    # Lấy giá trị trung vị (median) dự báo và làm tròn thành mã lỗi
    predicted_val = torch.median(forecast[0], dim=0).values[0].item()
    y_pred_chronos_b5.append(int(round(predicted_val)))

# 3. Tính toán kết quả
y_true_chronos_b5 = np.array(y_true_chronos_b5)
y_pred_chronos_b5 = np.clip(np.array(y_pred_chronos_b5), 0, df_chronos["alarm_code_cat"].max())

acc_chronos_b5 = np.mean(y_true_chronos_b5 == y_pred_chronos_b5)

print("\n==============================================================================")
print("📌 KẾT QUẢ MÔ HÌNH CHRONOS-T5 (TIME-SERIES FOUNDATION MODEL - AMAZON)")
print("==============================================================================")
print(f"🔹 Task B5 (Next Alarm Prediction) - Exact Accuracy : {acc_chronos_b5:.4f}")
print("==============================================================================")
